# Laboratory 08 — The wave equation

Module 07 left a chain of masses one step short of a string. Here you take the step: you watch
a chain of beads turn into a string as the beads multiply, time pulses between two photogates to
find out what sets their speed, check the solver against d'Alembert's exact solution, follow a
dust speck while a pulse runs through it, and push the time step past the Courant limit to watch
the computation destroy itself.

The last part is a measurement with an error bar, and it has a lesson in it about *which*
feature of a pulse you should call its arrival.

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell.

## Model specification

| | |
|---|---|
| **System** | the transverse displacement $y(x,t)$ of a uniform string of tension $T$ and linear density $\mu$ between two fixed ends |
| **Dynamics** | $\mu\,y_{tt} = T\,y_{xx}$, integrated by the leapfrog scheme and evaluated exactly by d'Alembert's solution |
| **Boundary** | fixed ends, kept out of the pulses' way except where the reflection is the point |
| **Ensemble** | deterministic; detector noise, where added, is Gaussian, independent and seeded |
| **Ignored** | stiffness, damping, gravity sag, longitudinal motion, nonlinearity |
| **Valid when** | slopes are small, $\lvert y_x \rvert \ll 1$; numerically, $S = v\,\Delta t/\Delta x \le 1$ with pulses resolved by many grid points |
| **Failure modes** | order-one slopes (the model breaks); $S > 1$ (the scheme breaks); grid dispersion on under-resolved features read as physics |

All the physics lives in `wavelab.waves` — open it and read it.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import coupled, measurement, waves
from wavelab.validation import scaling_exponent, seed_study

TENSION = 4.0  # N
MU = 0.01  # kg/m, mass per unit length
LENGTH = 1.0  # m, wall to wall
SPEED = waves.wave_speed(TENSION, MU)  # sqrt(T/mu): the medium's choice
MM = 1e3  # metres to millimetres, for plotting only

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2026)

print(f"v = sqrt(T/mu) = {SPEED} m/s")
print(f"a pulse crosses the {LENGTH} m string in {1e3 * LENGTH / SPEED:.0f} ms")

## Part 1 — From beads to a string

Module 07's chain, read sideways: $N$ beads on a taut thread. Each bead has mass $\mu a$, where
$a$ is the spacing, and each stretch of thread pulls its bead back toward the line with a
transverse force $T\,(y_{n+1} - y_n)/a$ for small slopes — a spring of stiffness $T/a$. That is
the dictionary $\mu = m/a$, $T = k_s a$ from the end of module 07, used in reverse.

Pluck the beads into a triangle and let go. The grey curve is the string itself, computed on
2001 points; the beads are solved exactly with `coupled.evolve`. Nothing is fitted.

In [ ]:
def triangle(s):
    """The pluck: 10 mm up at the centre, straight lines down to the walls."""
    return 0.01 * np.clip(1 - np.abs(s - LENGTH / 2) / (LENGTH / 2), 0, None)


CELLS = 2000
dx = LENGTH / CELLS
x = np.arange(CELLS + 1) * dx
# At S = 1 the solver carries a pluck exactly — part 6 is where you find out why — so this is
# the continuous string itself, sampled at 2001 points, 15 ms after release.
string = waves.simulate_string(
    triangle(x), np.zeros_like(x), dx, waves.cfl_max_dt(dx, SPEED), TENSION, MU, 600
)
t_look = string.times[-1]

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
bead_errors = {}
for axis, n in zip(axes, (5, 20, 100), strict=True):
    spacing = LENGTH / (n + 1)
    beads = np.arange(1, n + 1) * spacing
    chain = coupled.normal_mode_solve(*coupled.chain_matrices(n, MU * spacing, TENSION / spacing))
    shape = coupled.evolve(chain, triangle(beads), np.zeros(n), t_look)
    bead_errors[n] = np.max(np.abs(shape - np.interp(beads, x, string.y[-1]))) / 0.01

    axis.plot(x, MM * string.y[-1], color="0.6", lw=1.4, label="string")
    axis.plot(np.r_[0, beads, LENGTH], MM * np.r_[0, shape, 0], "o-", ms=3, lw=0.8,
              label=f"{n} beads")
    axis.set_xlabel("x (m)"), axis.legend(fontsize=8)
axes[0].set_ylabel("y (mm)")
plt.tight_layout()
plt.show()

for n, error in bead_errors.items():
    print(f"{n:3d} beads: worst disagreement with the string {error:.2%} of the pluck height")
print()
print("Roughly 1/N, not module 07's 1/N^2. A corner is made of every wavelength at once, and")
print("the short ones sit near the chain's band edge, where the chain is least like a string.")

## Part 2 — Predict

Before running the next cell, commit to answers.

1. Two pulses on the same string, one twice as tall as the other. Which reaches a photogate
   0.4 m away first?
2. You quadruple the tension. By what factor does the time to cross the string change?
3. A dust speck sits on the string. A pulse runs through it and on. Where is the speck
   afterwards?
4. You set the time step 1% longer than the Courant limit allows. What happens, and roughly
   when?

**Your prediction:**

1.
2.
3.
4.

## Part 3 — Photogates: what sets the speed

Two detectors, 0.4 m apart, record the displacement of the string where they sit. A pulse's
arrival at each is timed by the centroid of its record — a weighted average over every sample
of the pulse, not the single highest one — and the speed is the distance over the difference.

The time step is set once, for the fastest string in the sweep, and then left alone, as an
apparatus would be.

In [ ]:
GATE_LENGTH = 2.0
GATE_CELLS = 800
GATE_DX = GATE_LENGTH / GATE_CELLS
GATES = (480, 640)  # grid points at 1.2 m and 1.6 m
gate_x = np.arange(GATE_CELLS + 1) * GATE_DX
GATE_DISTANCE = (GATES[1] - GATES[0]) * GATE_DX


def gate_run(tension, mu, dt, height=0.01, width=0.05):
    """Pluck a 2 m string at 0.8 m and record the two gates until the pulse has passed both."""
    speed = waves.wave_speed(tension, mu)
    pluck = height * np.exp(-(((gate_x - 0.8) / width) ** 2))
    n_steps = int(np.ceil((0.8 + 8 * width) / speed / dt))
    run = waves.simulate_string(pluck, np.zeros_like(gate_x), GATE_DX, dt, tension, mu, n_steps)
    return run.times, run.y[:, GATES[0]], run.y[:, GATES[1]]


def speed_between_gates(times, first, second, half_window):
    passed = [measurement.pulse_arrival_time(times, record, half_window)
              for record in (first, second)]
    return GATE_DISTANCE / (passed[1] - passed[0])


tensions = np.geomspace(1.0, 10.0, 6)
dt = GATE_DX / waves.wave_speed(tensions.max(), MU)
speeds = []
for tension in tensions:
    times, first, second = gate_run(tension, MU, dt)
    half_window = 4 * 0.05 / waves.wave_speed(tension, MU)
    speeds.append(speed_between_gates(times, first, second, half_window))
speeds = np.array(speeds)

fig, (traces, fit) = plt.subplots(1, 2, figsize=(11, 3.2))
times, first, second = gate_run(4.0, MU, dt)
traces.plot(1e3 * times, MM * first, label="gate at 1.2 m")
traces.plot(1e3 * times, MM * second, label="gate at 1.6 m")
traces.set_xlabel("t (ms)"), traces.set_ylabel("y (mm)"), traces.legend(fontsize=8)
fit.loglog(tensions, speeds, "o", label="measured")
fit.loglog(tensions, np.sqrt(tensions / MU), lw=0.9, ls="--", color="0.5", label="sqrt(T/mu)")
fit.set_xlabel("tension (N)"), fit.set_ylabel("speed (m/s)"), fit.legend(fontsize=8)
plt.tight_layout()
plt.show()

exponent = scaling_exponent(tensions, speeds)
print(f"fitted exponent of v against T: {exponent:.6f}")
worst = np.max(np.abs(speeds / np.sqrt(tensions / MU) - 1))
print(f"worst disagreement with sqrt(T/mu): {worst:.1e}")
print()
print("Now race pulses that differ in the hand that made them, on one string at T = 4 N:")
race_dt = 0.5 * GATE_DX / SPEED
race = {}
for height in (0.005, 0.02):
    for width in (0.03, 0.08):
        times, first, second = gate_run(TENSION, MU, race_dt, height=height, width=width)
        race[height, width] = speed_between_gates(times, first, second, 4 * width / SPEED)
        print(f"  height {1e3 * height:4.0f} mm, width {100 * width:3.0f} cm: "
              f"{race[height, width]:.7f} m/s")

## Part 4 — Pluck and strike, against d'Alembert

Two ways to start a string. A **pluck** holds a shape still and lets go. A **strike** leaves the
string straight and gives part of it a velocity — here a rectangular one, $w = 0.5$ m/s over
the 0.4 m in the middle.

d'Alembert's solution says what each must do, exactly:

$$
y(x,t) = \tfrac12\left[y_0(x - vt) + y_0(x + vt)\right]
+ \frac{1}{2v}\left[W(x + vt) - W(x - vt)\right],
$$

with $W$ any antiderivative of the starting velocity. Solid lines are the solver, dashed lines
are the formula, on a 4 m string whose ends the pulses never reach.

In [ ]:
LONG = 4.0
LONG_CELLS = 1600
long_dx = LONG / LONG_CELLS
long_x = np.arange(LONG_CELLS + 1) * long_dx
long_dt = 0.5 * long_dx / SPEED
snapshots = [0, 40, 80]  # saved steps, 5 solver steps apart: t = 0, 12.5 ms, 25 ms


def hump(s):
    return 0.01 * np.exp(-(((s - 2.0) / 0.1) ** 2))


STRIKE = 0.5  # m/s, over 1.8 m < x < 2.2 m


def strike_velocity(s):
    return np.where((s > 1.8) & (s < 2.2), STRIKE, 0.0)


def strike_integral(s):
    return STRIKE * np.clip(s - 1.8, 0.0, 0.4)


pluck = waves.simulate_string(hump(long_x), np.zeros_like(long_x), long_dx, long_dt,
                              TENSION, MU, 400, save_every=5)
strike = waves.simulate_string(np.zeros_like(long_x), strike_velocity(long_x), long_dx, long_dt,
                               TENSION, MU, 400, save_every=5)
pluck_exact = waves.dalembert_solution(hump, SPEED, long_x, pluck.times)
strike_exact = waves.dalembert_solution(lambda s: np.zeros_like(s), SPEED, long_x, strike.times,
                                        v0_integral=strike_integral)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for step, colour in zip(snapshots, ("0.2", "tab:blue", "tab:orange"), strict=True):
    label = f"t = {1e3 * pluck.times[step]:.1f} ms"
    top.plot(long_x, MM * pluck.y[step], color=colour, lw=1.4, label=label)
    top.plot(long_x, MM * pluck_exact[step], color="k", lw=0.8, ls="--")
    bottom.plot(long_x, MM * strike.y[step], color=colour, lw=1.4, label=label)
    bottom.plot(long_x, MM * strike_exact[step], color="k", lw=0.8, ls="--")
top.set_ylabel("pluck y (mm)"), bottom.set_ylabel("strike y (mm)")
bottom.set_xlabel("x (m)"), top.legend(fontsize=8)
plt.tight_layout()
plt.show()

plateau = STRIKE * 0.2 / SPEED
print(f"pluck:  two halves of height {MM * pluck.y[-1].max():.4f} mm, from a 10 mm hump")
pluck_error = np.max(np.abs(pluck.y - pluck_exact)) / 0.01
print(f"        solver against d'Alembert: {pluck_error:.1e} of the height")
print(f"strike: plateau {MM * strike.y[-1, LONG_CELLS // 2]:.4f} mm; "
      f"d'Alembert's w l / v = {MM * plateau:.4f} mm")
print(f"        solver against d'Alembert: {np.max(np.abs(strike.y - strike_exact)) / plateau:.1e} "
      "of the plateau")

In [ ]:
# The strike's error sits at the corners of the trapezoid, and it is the grid's, not the
# string's: refine the grid at the same Courant number and watch it fall. Watch *how fast* it
# falls, too, and compare with the pluck's second order.
for cells in (800, 1600, 3200):
    grid_dx = LONG / cells
    grid_x = np.arange(cells + 1) * grid_dx
    grid_dt = 0.5 * grid_dx / SPEED
    steps = int(round(0.05 / grid_dt))
    run = waves.simulate_string(np.zeros_like(grid_x), strike_velocity(grid_x), grid_dx, grid_dt,
                                TENSION, MU, steps)
    exact = waves.dalembert_solution(lambda s: np.zeros_like(s), SPEED, grid_x, run.times[-1],
                                     v0_integral=strike_integral)
    print(f"{cells:5d} cells: worst error {np.max(np.abs(run.y[-1] - exact)) / plateau:.2e} "
          f"of the plateau, rms {np.sqrt(np.mean((run.y[-1] - exact) ** 2)) / plateau:.2e}")

## Part 5 — The falsifying experiment: follow a dust speck

Launch a single pulse moving right — a hump whose starting velocity is exactly what a
right-mover needs, $y_t = -v\,y_x$ — and mark one point of the string, 2 m along. Record where
that point is, and how fast it moves, while the pulse runs through it.

In [ ]:
def moving_hump(s):
    return 0.01 * np.exp(-(((s - 0.7) / 0.15) ** 2))


def moving_slope(s):
    return -2 * (s - 0.7) / 0.15**2 * moving_hump(s)


speck_run = waves.simulate_string(moving_hump(long_x), -SPEED * moving_slope(long_x), long_dx,
                                  long_dt, TENSION, MU, 2080, save_every=5)
speck = int(round(2.0 / long_dx))

fig, (along, record) = plt.subplots(1, 2, figsize=(11, 3.2))
for step in (0, 200, 416):
    (line,) = along.plot(long_x, MM * speck_run.y[step], lw=1.2,
                         label=f"t = {1e3 * speck_run.times[step]:.0f} ms")
    # The speck, where it actually is at that instant: always at x = 2 m, at whatever height.
    along.plot(long_x[speck], MM * speck_run.y[step, speck], "o", color=line.get_color(), ms=7,
               mec="crimson", mew=1.5)
along.axvline(long_x[speck], color="crimson", lw=0.6, ls=":")
along.set_xlabel("x (m)"), along.set_ylabel("y (mm)"), along.legend(fontsize=8)
record.plot(1e3 * speck_run.times, MM * speck_run.y[:, speck], color="crimson",
            label="speck height (mm)")
record.plot(1e3 * speck_run.times, speck_run.dydt[:, speck], color="0.4", lw=0.9,
            label="speck velocity (m/s)")
record.axhline(0, color="0.85", lw=0.8)
record.set_xlabel("t (ms)"), record.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"the pulse moved {SPEED * speck_run.times[-1]:.2f} m along the string")
print(f"the speck rose {MM * speck_run.y[:, speck].max():.3f} mm and finished at "
      f"{MM * speck_run.y[-1, speck]:.1e} mm — "
      f"{abs(speck_run.y[-1, speck]) / 0.01:.0e} of the pulse height, left by grid dispersion")
print(f"its x never changed: it is grid point {speck}, at {long_x[speck]} m, throughout")
print(f"its fastest transverse speed was {np.abs(speck_run.dydt[:, speck]).max():.3f} m/s; "
      f"the pulse travels at {SPEED} m/s")

## Part 6 — The Courant limit, and the magic step

The solver updates each point from its two neighbours:

$$
y_i^{n+1} = 2y_i^n - y_i^{n-1} + S^2\left(y_{i+1}^n - 2y_i^n + y_{i-1}^n\right),
\qquad S = \frac{v\,\Delta t}{\Delta x}.
$$

Run the same pluck at three Courant numbers. `simulate_string` refuses $S > 1$ unless it is
told that watching the failure is the point.

The right-hand panel does not plot the string. It plots how much of the string's shape is the
*sawtooth* — neighbouring points in exact antiphase, the shortest wave a grid can hold — as a
fraction of the pluck's height, on a logarithmic axis. Watch it from step 0, not from the moment
something visibly goes wrong.

In [ ]:
cells = 400
short_dx = LENGTH / cells
short_x = np.arange(cells + 1) * short_dx


def gaussian_pluck(s):
    return 0.01 * np.exp(-(((s - 0.5) / 0.05) ** 2))


try:
    waves.simulate_string(gaussian_pluck(short_x), np.zeros_like(short_x), short_dx,
                          1.01 * short_dx / SPEED, TENSION, MU, 10)
except ValueError as refusal:
    print(f"refused: {refusal}")
print()

def sawtooth(y):
    """Amplitude of the shortest wave the grid can hold — neighbours in exact antiphase."""
    points = np.arange(y.shape[-1])
    cells = points.size - 1
    pattern = (-1.0) ** (points + 1) * np.sin(np.pi * points / cells)
    return np.abs(y @ pattern) * 2 / cells


fig, (strings, growth) = plt.subplots(1, 2, figsize=(11, 3.4))
courant_runs = {}
for courant in (0.9, 1.0, 1.01):
    run = waves.simulate_string(gaussian_pluck(short_x), np.zeros_like(short_x), short_dx,
                                courant * short_dx / SPEED, TENSION, MU, 150,
                                allow_unstable=True)
    courant_runs[courant] = run
    peak = np.max(np.abs(run.y), axis=1)
    # Drawn as the largest value so far: at the rounding floor the projection hits exactly zero
    # every few steps, which a log axis would draw as a spike.
    growth.semilogy(np.arange(peak.size), np.maximum.accumulate(sawtooth(run.y)) / 0.01,
                    label=f"S = {courant}")
    strings.plot(short_x, np.clip(MM * run.y[-1], -15, 15), lw=1.0, label=f"S = {courant}")
    beyond = np.nonzero(peak > 0.1)[0]
    print(f"S = {courant}: largest |y| after 150 steps {MM * peak[-1]:.3g} mm"
          + (f"; passed ten times the pluck height at step {beyond[0]}" if beyond.size else ""))
strings.set_xlabel("x (m)"), strings.set_ylabel("y (mm), clipped at 15")
growth.set_xlabel("step"), growth.set_ylabel("sawtooth / pluck height")
growth.set_ylim(1e-18, 1e3)
strings.legend(fontsize=8), growth.legend(fontsize=8)
plt.tight_layout()
plt.show()

# d'Alembert's solution has no walls, so compare only while the halves are still seven widths
# clear of them: the first 60 steps.
early = slice(0, 61)
for courant in (0.9, 1.0):
    run = courant_runs[courant]
    exact = waves.dalembert_solution(gaussian_pluck, SPEED, short_x, run.times[early])
    error = np.max(np.abs(run.y[early] - exact)) / 0.01
    print(f"S = {courant}: largest error against d'Alembert over 60 steps {error:.1e} "
          "of the pluck height")

## Part 7 — Measurement, with an error bar

Back to the photogates, now with detector noise: each gate's record gets independent Gaussian
noise of 5% of the passing pulse's height. Measure $v$ across seeds and quote it as a number
with an uncertainty.

Time the pulses two ways: by the centroid, as in part 3, and by reading off the top of each
record. Both are honest arithmetic on the same data. Run the cell, then compare the two error
bars.

In [ ]:
times, first, second = gate_run(TENSION, MU, 0.5 * GATE_DX / SPEED)
NOISE = 0.05 * 0.005  # 5% of the 5 mm pulse that passes each gate
half_window = 4 * 0.05 / SPEED


def peak_time(record):
    """The highest sample, refined between samples by a three-point parabola."""
    i = int(np.argmax(record))
    before, top, after = record[i - 1 : i + 2]
    step = times[1] - times[0]
    return times[i] + 0.5 * (before - after) / (before - 2 * top + after) * step


def by_centroid(generator):
    noisy = [measurement.add_noise(record, NOISE, generator) for record in (first, second)]
    return speed_between_gates(times, *noisy, half_window)


def by_peak(generator):
    noisy = [measurement.add_noise(record, NOISE, generator) for record in (first, second)]
    return GATE_DISTANCE / (peak_time(noisy[1]) - peak_time(noisy[0]))


centroid = seed_study(by_centroid, n_seeds=32, base_seed=8)
peak = seed_study(by_peak, n_seeds=32, base_seed=8)

fig, axis = plt.subplots(figsize=(8, 3))
noisy_first = measurement.add_noise(first, NOISE, rng)
axis.plot(1e3 * times, MM * noisy_first, lw=0.7, color="0.5", label="gate at 1.2 m, noisy")
axis.plot(1e3 * times, MM * first, lw=1.2, color="k", label="the pulse underneath")
axis.set_xlabel("t (ms)"), axis.set_ylabel("y (mm)"), axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

std_centroid = centroid.relative_spread * centroid.mean
std_peak = peak.relative_spread * peak.mean
print(f"by centroid: v = ({centroid.mean:.2f} +/- {centroid.standard_error:.2f}) m/s, "
      f"single-run scatter {std_centroid:.2f} m/s")
print(f"by peak    : v = ({peak.mean:.2f} +/- {peak.standard_error:.2f}) m/s, "
      f"single-run scatter {std_peak:.2f} m/s")
print(f"sqrt(T/mu) : {SPEED} m/s")
print()
print(f"The peak's scatter is {std_peak / std_centroid:.1f} times the centroid's. Same data, same")
print("noise: the peak is decided by the three highest samples, the centroid by every sample")
print("of the pulse.")

In [ ]:
# 1. Beads become a string: the disagreement shrinks with N and is under 1% at N = 100.
assert bead_errors[5] > bead_errors[20] > bead_errors[100]
assert bead_errors[100] < 0.01

# 2. The medium sets the speed: v goes as T^(1/2), whatever the hand did.
assert abs(exponent - 0.5) < 1e-3
assert max(race.values()) - min(race.values()) < 1e-5 * SPEED

# 3. d'Alembert: a pluck splits into two halves.
assert abs(pluck.y[-1].max() - 0.005) < 1e-5

# 4. The falsifier: the speck comes back.
assert abs(speck_run.y[-1, speck]) < 1e-5 * 0.01

# 5. Past S = 1 the scheme explodes; at S = 1 a pluck is exact.
assert np.max(np.abs(courant_runs[1.01].y[-1])) > 1.0
assert np.max(np.abs(courant_runs[0.9].y)) <= 0.01 * (1 + 1e-9)
exact = waves.dalembert_solution(gaussian_pluck, SPEED, short_x, courant_runs[1.0].times[early])
assert np.max(np.abs(courant_runs[1.0].y[early] - exact)) < 1e-13

# 6. The noisy measurement agrees with sqrt(T/mu) within its uncertainty.
assert centroid.agrees_with(SPEED, n_sigma=3.0)

print("all checks passed")
print(f"v measured through noise: ({centroid.mean:.2f} +/- {centroid.standard_error:.2f}) m/s "
      f"against sqrt(T/mu) = {SPEED} m/s")

## Part 8 — Explore it yourself

Launch a pulse and time it. Three things are worth hunting for.

- Change the height and the width and watch the measured speed. Then change $T$ or $\mu$.
- Make the pulse narrow — a few grid spacings wide — at $S = 0.5$, and look for a wake of ripples
  behind it. Then set $S = 1$ and look again.
- Push $S$ just past 1 and find out how many steps the string survives.

In [ ]:
import ipywidgets as widgets


def explore(tension=4.0, mu=0.01, height_mm=10.0, width_cm=5.0, courant=0.5):
    speed = waves.wave_speed(tension, mu)
    dt = courant * GATE_DX / speed
    width = width_cm / 100
    pluck = (height_mm / MM) * np.exp(-(((gate_x - 0.8) / width) ** 2))
    n_steps = int(np.ceil((0.8 + 8 * width) / speed / dt))
    run = waves.simulate_string(pluck, np.zeros_like(gate_x), GATE_DX, dt, tension, mu, n_steps,
                                allow_unstable=True)

    fig, (shape_axis, gate_axis) = plt.subplots(1, 2, figsize=(11, 3.2))
    shape_axis.plot(gate_x, np.clip(MM * run.y[-1], -2 * height_mm, 2 * height_mm), lw=1.0)
    for gate in GATES:
        shape_axis.axvline(gate_x[gate], color="crimson", lw=0.7, ls=":")
    shape_axis.set_xlabel("x (m)"), shape_axis.set_ylabel("y (mm)")
    gate_axis.plot(1e3 * run.times, MM * run.y[:, GATES[0]], label="gate at 1.2 m")
    gate_axis.plot(1e3 * run.times, MM * run.y[:, GATES[1]], label="gate at 1.6 m")
    gate_axis.set_xlabel("t (ms)"), gate_axis.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    if not np.all(np.isfinite(run.y)) or np.max(np.abs(run.y)) > 10 * height_mm / MM:
        print(f"S = {courant}: the computation blew up — nothing below is a measurement")
        return
    measured = speed_between_gates(run.times, run.y[:, GATES[0]], run.y[:, GATES[1]],
                                   4 * width / speed)
    print(f"measured {measured:.5f} m/s; sqrt(T/mu) = {speed:.5f} m/s")


widgets.interact_manual(
    explore,
    tension=widgets.FloatSlider(min=1.0, max=10.0, step=0.5, value=4.0, description="T (N)"),
    mu=widgets.FloatLogSlider(base=10, min=-3, max=-1, step=0.1, value=0.01,
                              description="mu (kg/m)"),
    height_mm=widgets.FloatSlider(min=1.0, max=20.0, step=1.0, value=10.0,
                                  description="height (mm)"),
    width_cm=widgets.FloatSlider(min=0.5, max=10.0, step=0.5, value=5.0, description="width (cm)"),
    courant=widgets.FloatSlider(min=0.2, max=1.05, step=0.01, value=0.5, description="S"),
);

## Check your understanding

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "08-wave-equation.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. In part 3 the height and width of a pulse made no difference to its speed, and the tension
   did. Explain why, to someone who believes that flicking a rope harder sends the flick along
   it faster. Your explanation should say what the hand controls and what the rope controls.
2. In part 5 the speck finished where it started. So what crossed the string? Say what you would
   point at in a photograph of the string to show someone the thing that moves.
3. In part 6 nothing about the string changed between $S = 1.0$ and $S = 1.01$. Say what did
   change, and why a physicist who saw only the $S = 1.01$ plot would be wrong to conclude the
   string is unstable.

**Your answers:**

1.
2.
3.